10. Regular Expression Matching

Given an input string s and a pattern p, implement regular expression matching with support for '.' and '*' where:

- '.' Matches any single character.
- '*' Matches zero or more of the preceding element.
Return a boolean indicating whether the matching covers the entire input string (not partial).

 

Example 1:

- Input: s = "aa", p = "a"
- Output: false
- Explanation: "a" does not match the entire string "aa".

Example 2:

- Input: s = "aa", p = "a*"
- Output: true
- Explanation: '*' means zero or more of the preceding element, 'a'. Therefore, by repeating 'a' once, it becomes "aa".

Example 3:

- Input: s = "ab", p = ".*"
- Output: true
- Explanation: ".*" means "zero or more (*) of any character (.)".
 

Constraints:

- 1 <= s.length <= 20
- 1 <= p.length <= 20
- s contains only lowercase English letters.
- p contains only lowercase English letters, '.', and '*'.
- It is guaranteed for each appearance of the character '*', there will be a previous valid character to match.

## problem statement

here we are given with both s and p
- s => only english lower case
- p => only english , . , *

- "." -> any any single character
- "*" -> any character which zero or more

that menas , if s have any characters and just .* then it is true
- if s have only one chareacter "a" then "." is true
- if s have "aa" then "a*" is true since there zero or more "a"
- if s have "a" and the p is "ab*" true because the b here is zero or more because of *
- if s have "ab" and  p is "a." it matches since "." cab be any character


## Solution Approach

This is a Dynamic Programming problem.

dp[i][j] = True if s[0:i] matches p[0:j]

Key cases:
- If pattern has '*', we can use it 0 times or 1+ times
- If current chars match (or pattern has '.'), move both forward
- Handle empty string and pattern edge cases


## Approach 1: Bottom-Up Dynamic Programming (Tabulation)

Time: O(m*n), Space: O(m*n)

In [ ]:
class Solution1:
    def isMatch(self, s: str, p: str) -> bool:
        # DP table: dp[i][j] = does s[0:i] match p[0:j]
        dp = [[False] * (len(p) + 1) for _ in range(len(s) + 1)]
        
        # Base case: empty string matches empty pattern
        dp[0][0] = True
        
        # Handle patterns like a*, a*b*, a*b*c* that can match empty string
        for j in range(2, len(p) + 1):
            if p[j - 1] == '*':
                dp[0][j] = dp[0][j - 2]
        
        # Fill the DP table
        for i in range(1, len(s) + 1):
            for j in range(1, len(p) + 1):
                s_char = s[i - 1]
                p_char = p[j - 1]
                
                if p_char == '*':
                    # '*' matches zero or more of preceding element
                    prev_p_char = p[j - 2]
                    
                    # Case 1: Use * zero times (ignore prev_char and *)
                    dp[i][j] = dp[i][j - 2]
                    
                    # Case 2: Use * one or more times
                    if prev_p_char == s_char or prev_p_char == '.':
                        dp[i][j] = dp[i][j] or dp[i - 1][j]
                
                elif p_char == '.' or p_char == s_char:
                    # Characters match or pattern has '.'
                    dp[i][j] = dp[i - 1][j - 1]
        
        return dp[len(s)][len(p)]


# Test
sol1 = Solution1()
print("Approach 1 - DP Tabulation:")
print(sol1.isMatch("aa", "a"))          # False
print(sol1.isMatch("aa", "a*"))         # True
print(sol1.isMatch("ab", ".*"))         # True
print(sol1.isMatch("aab", "c*a*b"))     # True

## Approach 2: Top-Down Recursion with Memoization

Time: O(m*n), Space: O(m*n)

In [ ]:
class Solution2:
    def isMatch(self, s: str, p: str) -> bool:
        memo = {}
        
        def dp(i, j):
            # i = index in s, j = index in p
            if (i, j) in memo:
                return memo[(i, j)]
            
            # Base case: pattern exhausted
            if j == len(p):
                return i == len(s)
            
            # Check if current characters match
            first_match = i < len(s) and (p[j] == s[i] or p[j] == '.')
            
            # Check if next character is '*'
            if j + 1 < len(p) and p[j + 1] == '*':
                # Case 1: Use * zero times (skip pattern[j] and *)
                # Case 2: Use * one or more times (if first_match, consume s[i])
                result = dp(i, j + 2) or (first_match and dp(i + 1, j))
            else:
                # No '*', just match current and move forward
                result = first_match and dp(i + 1, j + 1)
            
            memo[(i, j)] = result
            return result
        
        return dp(0, 0)


# Test
sol2 = Solution2()
print("\nApproach 2 - Recursion with Memoization:")
print(sol2.isMatch("aa", "a"))          # False
print(sol2.isMatch("aa", "a*"))         # True
print(sol2.isMatch("ab", ".*"))         # True
print(sol2.isMatch("aab", "c*a*b"))     # True

## Approach 3: Space-Optimized DP (Using 2 Rows)

Time: O(m*n), Space: O(n)

In [ ]:
class Solution3:
    def isMatch(self, s: str, p: str) -> bool:
        m, n = len(s), len(p)
        
        # Use only 2 rows instead of full table
        prev = [False] * (n + 1)
        curr = [False] * (n + 1)
        
        # Base case
        prev[0] = True
        
        # Initialize first row (empty string matching pattern)
        for j in range(2, n + 1):
            if p[j - 1] == '*':
                prev[j] = prev[j - 2]
        
        # Fill row by row
        for i in range(1, m + 1):
            curr[0] = False  # Non-empty string can't match empty pattern
            
            for j in range(1, n + 1):
                s_char = s[i - 1]
                p_char = p[j - 1]
                
                if p_char == '*':
                    prev_p_char = p[j - 2]
                    curr[j] = curr[j - 2]  # Use * zero times
                    
                    if prev_p_char == s_char or prev_p_char == '.':
                        curr[j] = curr[j] or prev[j]  # Use * one or more times
                
                elif p_char == '.' or p_char == s_char:
                    curr[j] = prev[j - 1]
                else:
                    curr[j] = False
            
            # Swap rows
            prev, curr = curr, prev
        
        return prev[n]


# Test
sol3 = Solution3()
print("\nApproach 3 - Space-Optimized DP:")
print(sol3.isMatch("aa", "a"))          # False
print(sol3.isMatch("aa", "a*"))         # True
print(sol3.isMatch("ab", ".*"))         # True
print(sol3.isMatch("aab", "c*a*b"))     # True

## All Test Cases

In [ ]:
# Comprehensive test cases
test_cases = [
    ("aa", "a", False),
    ("aa", "a*", True),
    ("ab", ".*", True),
    ("aab", "c*a*b", True),
    ("mississippi", "mis*is*p*.", False),
    ("", "a*", True),
    ("a", "ab*", True),
    ("ab", ".*c", False),
    ("aaa", "a*a", True),
    ("aaa", "ab*a*c*a", True),
]

print("\nComprehensive Test Results:")
print("="*50)
sol = Solution1()  # Use any approach

for s, p, expected in test_cases:
    result = sol.isMatch(s, p)
    status = "✓" if result == expected else "✗"
    print(f"{status} s='{s}', p='{p}' => {result} (expected: {expected})")